# Reformat and Cleaning

Notebook ini fokus ke tahap normalisasi data multi-daerah.

Tujuan utama:
- standarisasi schema
- bersihkan bad rows
- imputasi dengan konteks daerah
- export satu dataset bersih yang siap dipakai training

Catatan desain:
- tidak ada train/test split di notebook ini
- split final dilakukan di notebook training per `nama_daerah`


## 1. Define Library and Path

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

raw_path = Path('../data/csv/master_data_ocds.csv')
cleaned_root = Path('../data/cleaned')
csv_root = Path('../data/csv')

cleaned_data_path = cleaned_root / 'cleaned_data.csv'
normalized_data_path = csv_root / 'normalized_data.csv'
summary_path = cleaned_root / 'cleaned_data_summary_by_daerah.csv'

cleaned_root.mkdir(parents=True, exist_ok=True)
csv_root.mkdir(parents=True, exist_ok=True)


## 2. Load Raw Data

In [2]:
data = pd.read_csv(raw_path)

print(f'Loaded raw data: {len(data):,} rows, {len(data.columns)} columns')
print('\nInitial columns:')
print(data.columns.tolist())
print('\nInitial missing values:')
print(data.isnull().sum().to_string())

display(data.head(3))


Loaded raw data: 21,156 rows, 20 columns

Initial columns:
['lspe_id', 'nama_daerah', 'source_year', 'source_file', 'ocid', 'release_id', 'date', 'buyer_name', 'buyer_id', 'tender_id', 'tender_title', 'mainProcurementCategory', 'tender_minValue', 'tender_datePublished', 'tender_status', 'award_id', 'award_title', 'award_date', 'award_value', 'award_supplier']

Initial missing values:
lspe_id                       0
nama_daerah                   0
source_year                   0
source_file                   0
ocid                          0
release_id                    0
date                          0
buyer_name                 3438
buyer_id                   2026
tender_id                     0
tender_title                  0
mainProcurementCategory       0
tender_minValue               0
tender_datePublished          0
tender_status                 0
award_id                      0
award_title                   0
award_date                    0
award_value                   0
award

,lspe_id,nama_daerah,source_year,source_file,ocid,release_id,date,buyer_name,buyer_id,tender_id,tender_title,mainProcurementCategory,tender_minValue,tender_datePublished,tender_status,award_id,award_title,award_date,award_value,award_supplier
0,127,Jakarta,2015,ocds-data-tender-2015-127.json,ocds-20h3g7-26650127,ocds-20h3g7-26650127-1,2014-12-24T00:00:00.000000Z,Pemerintah Daerah Provinsi DKI Jakarta,D69,26650127,Pengadaan Makanan Binatang/Satwa TMR (pakan ke...,goods,1.499502e+09,2014-12-08T00:00:00.000000Z,complete,26650127-1,Pengadaan Makanan Binatang/Satwa TMR (pakan ke...,2014-12-24T00:00:00.000000Z,1.157888e+09,CV. USAHA SELARAS
1,127,Jakarta,2015,ocds-data-tender-2015-127.json,ocds-20h3g7-26651127,ocds-20h3g7-26651127-2,2015-04-24T00:00:00.000000Z,Pemerintah Daerah Provinsi DKI Jakarta,D69,26651127,Pengadaan Makanan Binatang/ Satwa TMR (Buah da...,goods,9.039069e+09,2014-12-30T00:00:00.000000Z,complete,26651127-1,Pengadaan Makanan Binatang/ Satwa TMR (Buah da...,2015-04-24T00:00:00.000000Z,6.867945e+09,CV. PANGAN INDO
2,127,Jakarta,2015,ocds-data-tender-2015-127.json,ocds-20h3g7-26652127,ocds-20h3g7-26652127-2,2015-04-23T00:00:00.000000Z,Pemerintah Daerah Provinsi DKI Jakarta,D69,26652127,Belanja Bahan Pakan Daging/Binatang Hidup,goods,6.271924e+09,2014-12-31T00:00:00.000000Z,complete,26652127-1,Belanja Bahan Pakan Daging/Binatang Hidup,2015-04-23T00:00:00.000000Z,6.196783e+09,CV. ROSADIA PETAHO


## 3. Standardize Column Names and Parse Types

In [3]:
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')

date_cols = ['date', 'tender_datepublished', 'award_date']
for col in date_cols:
    data[col] = pd.to_datetime(data[col], utc=True, errors='coerce').dt.tz_localize(None)

num_cols = ['tender_minvalue', 'award_value']
for col in num_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

print('Columns after normalization:')
print(data.columns.tolist())
print('\nDtypes after parsing:')
print(data[date_cols + num_cols].dtypes.to_string())
print('\nMissing dates after parsing:')
print(data[date_cols].isnull().sum().to_string())


Columns after normalization:
['lspe_id', 'nama_daerah', 'source_year', 'source_file', 'ocid', 'release_id', 'date', 'buyer_name', 'buyer_id', 'tender_id', 'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_datepublished', 'tender_status', 'award_id', 'award_title', 'award_date', 'award_value', 'award_supplier']

Dtypes after parsing:
date                    datetime64[us]
tender_datepublished    datetime64[us]
award_date              datetime64[us]
tender_minvalue                float64
award_value                    float64

Missing dates after parsing:
date                    0
tender_datepublished    0
award_date              0


## 4. Remove Bad Rows

In [4]:
before = len(data)
data.drop_duplicates(inplace=True)
after_duplicates = len(data)

print(f'Duplicate rows removed: {before - after_duplicates:,}')

required_cols = ['lspe_id', 'nama_daerah', 'award_value', 'award_supplier', 'award_date', 'date']
before_required = len(data)
data.dropna(subset=required_cols, inplace=True)
after_required = len(data)

print(f'Rows removed for required nulls: {before_required - after_required:,}')

before_valid_dates = len(data)
data.dropna(subset=['date', 'award_date'], inplace=True)
after_valid_dates = len(data)

print(f'Rows removed for invalid parsed date/award_date: {before_valid_dates - after_valid_dates:,}')

before_positive = len(data)
data = data[data['award_value'] > 0].copy()
after_positive = len(data)

print(f'Rows removed for non-positive award_value: {before_positive - after_positive:,}')
print(f'Rows remaining after bad-row cleanup: {len(data):,}')


Duplicate rows removed: 0
Rows removed for required nulls: 0
Rows removed for invalid parsed date/award_date: 0
Rows removed for non-positive award_value: 0
Rows remaining after bad-row cleanup: 21,156


## 5. Basic Multi-Daerah Standardization

In [5]:
data['lspe_id'] = data['lspe_id'].astype(str).str.strip()
data['nama_daerah'] = data['nama_daerah'].astype(str).str.strip().str.title()

print('Rows per daerah before imputation:')
print(data.groupby(['nama_daerah', 'lspe_id']).size().to_string())


Rows per daerah before imputation:
nama_daerah  lspe_id
Jakarta      127        8678
Jawa Tengah  42         6038
Jawa Timur   15         6440


## 6. Recover Missing Buyer Name with Region Context

In [6]:
# buyer_name: try to recover from other rows sharing the same buyer_id within the same daerah
buyer_map = (
    data.dropna(subset=['buyer_name', 'buyer_id'])
        .drop_duplicates(['nama_daerah', 'buyer_id'])
        .set_index(['nama_daerah', 'buyer_id'])['buyer_name']
        .to_dict()
)

data['buyer_name'] = data['buyer_name'].fillna(
    data.apply(lambda row: buyer_map.get((row['nama_daerah'], row['buyer_id'])), axis=1)
)

# Remaining buyer_name nulls fill with region-specific province label
data['buyer_name'] = data['buyer_name'].fillna(
    'Pemerintah Daerah Provinsi ' + data['nama_daerah']
)

print('Missing buyer_name after recovery:')
print(int(data['buyer_name'].isnull().sum()))
print('\nSample buyer names:')
print(data['buyer_name'].value_counts().head(10).to_string())


Missing buyer_name after recovery:
0

Sample buyer names:
buyer_name
Pemerintah Daerah Provinsi DKI Jakarta    7136
Pemerintah Daerah Provinsi Jawa Timur     6165
Pemerintah Daerah Provinsi Jawa Tengah    5894
Pemerintah Daerah Provinsi Jakarta        1481
Badan Pengembangan Wilayah Suramadu        100
Badan Penanggulangan Lumpur Sidoarjo        54
Komite Olah Raga Nasional Indonesia         52
Sekretariat Kabinet                         35
Kementerian Kesehatan                       33
PT Surabaya Industrial Estate Rungkut       33


## 7. Fill Remaining Missing Values

In [7]:
# mainprocurementcategory fill with 'Unknown Category'
data['mainprocurementcategory'] = data['mainprocurementcategory'].fillna('Unknown Category')

# tender_status fill with 'Unknown Status'
data['tender_status'] = data['tender_status'].fillna('Unknown Status')

# tender_title and award_title fill
data['tender_title'] = data['tender_title'].fillna('Unknown Tender Title')
data['award_title'] = data['award_title'].fillna(data['tender_title'])

# tender_minvalue fill missing with median per daerah first, then global median
median_min_by_region = data.groupby('nama_daerah')['tender_minvalue'].transform('median')
global_median_min = data['tender_minvalue'].median()
data['tender_minvalue'] = data['tender_minvalue'].fillna(median_min_by_region)
data['tender_minvalue'] = data['tender_minvalue'].fillna(global_median_min)

print('Missing values after general imputation:')
print(data[['mainprocurementcategory', 'tender_status', 'tender_title', 'award_title', 'tender_minvalue']].isnull().sum().to_string())


Missing values after general imputation:
mainprocurementcategory    0
tender_status              0
tender_title               0
award_title                0
tender_minvalue            0


## 8. Build Temporal and Ratio Features

In [8]:
data['days_to_award'] = (data['award_date'] - data['tender_datepublished']).dt.days

# fallback jika tender_datepublished kosong: gunakan selisih award_date dengan release date
fallback_days = (data['award_date'] - data['date']).dt.days
data['days_to_award'] = data['days_to_award'].fillna(fallback_days)
data['days_to_award'] = data['days_to_award'].fillna(0).clip(lower=0)

# budget_utilization_ratio: award_value dibanding tender_minvalue
ratio_denominator = data['tender_minvalue'].replace(0, np.nan)
data['budget_utilization_ratio'] = (
    data['award_value'] / ratio_denominator
).clip(lower=0, upper=10)

ratio_median_by_region = data.groupby('nama_daerah')['budget_utilization_ratio'].transform('median')
global_ratio_median = data['budget_utilization_ratio'].median()
data['budget_utilization_ratio'] = data['budget_utilization_ratio'].fillna(ratio_median_by_region)
data['budget_utilization_ratio'] = data['budget_utilization_ratio'].fillna(global_ratio_median)
data['budget_utilization_ratio'] = data['budget_utilization_ratio'].fillna(1.0)

fallback_days_count = int(data['tender_datepublished'].isnull().sum())

print('Temporal feature stats:')
print(data[['days_to_award']].describe().to_string())
print('\nRatio feature stats:')
print(data[['budget_utilization_ratio']].describe().to_string())
print(f'\nRows using fallback days_to_award logic: {fallback_days_count:,}')


Temporal feature stats:
       days_to_award
count   21156.000000
mean       28.138779
std       186.084554
min         0.000000
25%        15.000000
50%        22.000000
75%        35.000000
max     19156.000000

Ratio feature stats:
       budget_utilization_ratio
count              21156.000000
mean                   0.883579
std                    0.097605
min                    0.040056
25%                    0.820036
50%                    0.903046
75%                    0.958601
max                    1.997665

Rows using fallback days_to_award logic: 0


## 9. Clean Text Columns

In [9]:
text_cols = [
    'buyer_name', 'tender_title', 'mainprocurementcategory',
    'tender_status', 'award_title', 'award_supplier'
]

for col in text_cols:
    data[col] = (
        data[col].astype(str)
            .str.strip()
            .str.replace(r'\s+', ' ', regex=True)
            .str.title()
    )

print('Sample procurement categories:')
print(data['mainprocurementcategory'].value_counts().head(10).to_string())


Sample procurement categories:
mainprocurementcategory
Services    8246
Works       7806
Goods       5104


## 10. Drop Technical Identity Columns

In [10]:
cols_to_drop = ['ocid', 'release_id', 'buyer_id', 'award_id', 'tender_id']
data.drop(columns=[col for col in cols_to_drop if col in data.columns], inplace=True)

print('Remaining columns:')
print(data.columns.tolist())


Remaining columns:
['lspe_id', 'nama_daerah', 'source_year', 'source_file', 'date', 'buyer_name', 'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_datepublished', 'tender_status', 'award_title', 'award_date', 'award_value', 'award_supplier', 'days_to_award', 'budget_utilization_ratio']


## 11. Reorder Columns and Build Summary

In [11]:
column_order = [
    'lspe_id', 'nama_daerah', 'source_year', 'source_file', 'date', 'buyer_name',
    'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_datepublished',
    'tender_status', 'award_title', 'award_date', 'award_value', 'award_supplier',
    'days_to_award', 'budget_utilization_ratio'
]

remaining_columns = [col for col in data.columns if col not in column_order]
data = data[[col for col in column_order if col in data.columns] + remaining_columns]

summary_df = (
    data.groupby(['nama_daerah', 'lspe_id'])
        .agg(
            rows=('nama_daerah', 'size'),
            min_award_date=('award_date', 'min'),
            max_award_date=('award_date', 'max'),
            median_award_value=('award_value', 'median'),
            median_tender_minvalue=('tender_minvalue', 'median'),
            median_days_to_award=('days_to_award', 'median')
        )
        .reset_index()
        .sort_values(['nama_daerah', 'lspe_id'])
        .reset_index(drop=True)
)

print('Per-daerah cleaned summary:')
display(summary_df)


Per-daerah cleaned summary:


,nama_daerah,lspe_id,rows,min_award_date,max_award_date,median_award_value,median_tender_minvalue,median_days_to_award
0,Jakarta,127,8678,2014-12-24,2023-10-18 00:00:00,7.585620e+08,8.799827e+08,21.0
1,Jawa Tengah,42,6038,2014-11-29,2023-10-16 00:00:00,6.520898e+08,7.478834e+08,25.0
2,Jawa Timur,15,6440,2014-12-19,2024-10-02 15:00:00,5.808605e+08,6.496555e+08,18.0


## 12. Save Output Files

In [12]:
data.to_csv(cleaned_data_path, index=False)
data.to_csv(normalized_data_path, index=False)
summary_df.to_csv(summary_path, index=False)

print('FINAL DATASET SUMMARY')
print('=' * 50)
print(f'Total rows   : {len(data):,}')
print(f'Total columns: {len(data.columns)}')
print('\nMissing values after cleaning:')
print(data.isnull().sum().to_string())
print('\nSaved files:')
print(f'- Cleaned dataset   : {cleaned_data_path.resolve()}')
print(f'- Normalized dataset: {normalized_data_path.resolve()}')
print(f'- Region summary    : {summary_path.resolve()}')

display(data.head(5))


FINAL DATASET SUMMARY
Total rows   : 21,156
Total columns: 17

Missing values after cleaning:
lspe_id                     0
nama_daerah                 0
source_year                 0
source_file                 0
date                        0
buyer_name                  0
tender_title                0
mainprocurementcategory     0
tender_minvalue             0
tender_datepublished        0
tender_status               0
award_title                 0
award_date                  0
award_value                 0
award_supplier              0
days_to_award               0
budget_utilization_ratio    0

Saved files:
- Cleaned dataset   : C:\Users\VICTUS\coding\collab\3\data\cleaned\cleaned_data.csv
- Normalized dataset: C:\Users\VICTUS\coding\collab\3\data\csv\normalized_data.csv
- Region summary    : C:\Users\VICTUS\coding\collab\3\data\cleaned\cleaned_data_summary_by_daerah.csv


,lspe_id,nama_daerah,source_year,source_file,date,buyer_name,tender_title,mainprocurementcategory,tender_minvalue,tender_datepublished,tender_status,award_title,award_date,award_value,award_supplier,days_to_award,budget_utilization_ratio
0,127,Jakarta,2015,ocds-data-tender-2015-127.json,2014-12-24,Pemerintah Daerah Provinsi Dki Jakarta,Pengadaan Makanan Binatang/Satwa Tmr (Pakan Ke...,Goods,1.499502e+09,2014-12-08,Complete,Pengadaan Makanan Binatang/Satwa Tmr (Pakan Ke...,2014-12-24,1.157888e+09,Cv. Usaha Selaras,16,0.772182
1,127,Jakarta,2015,ocds-data-tender-2015-127.json,2015-04-24,Pemerintah Daerah Provinsi Dki Jakarta,Pengadaan Makanan Binatang/ Satwa Tmr (Buah Da...,Goods,9.039069e+09,2014-12-30,Complete,Pengadaan Makanan Binatang/ Satwa Tmr (Buah Da...,2015-04-24,6.867945e+09,Cv. Pangan Indo,115,0.759807
2,127,Jakarta,2015,ocds-data-tender-2015-127.json,2015-04-23,Pemerintah Daerah Provinsi Dki Jakarta,Belanja Bahan Pakan Daging/Binatang Hidup,Goods,6.271924e+09,2014-12-31,Complete,Belanja Bahan Pakan Daging/Binatang Hidup,2015-04-23,6.196783e+09,Cv. Rosadia Petaho,113,0.988019
3,127,Jakarta,2015,ocds-data-tender-2015-127.json,2015-04-15,Pemerintah Daerah Provinsi Dki Jakarta,Penyelenggaraan Kegiatan Asia Golf Tourism Con...,Services,1.988607e+10,2015-03-31,Complete,Penyelenggaraan Kegiatan Asia Golf Tourism Con...,2015-04-15,1.934999e+10,Pt. Ekspo Kreatif Indo,15,0.973043
4,127,Jakarta,2015,ocds-data-tender-2015-127.json,2015-04-17,Pemerintah Daerah Provinsi Dki Jakarta,Pengadaan Naskah Soal Ujian Sekolah/ Madrasah ...,Services,1.612680e+09,2015-04-02,Complete,Pengadaan Naskah Soal Ujian Sekolah/ Madrasah ...,2015-04-17,1.263166e+09,Pt Pura Barutama,15,0.783271
